In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage

load_dotenv()

/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()
prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant.\n\n"
     "Summary of conversation so far:\n{summary}"),
    ("human","{input}")])
chain=prompt|llm|parser
# ── Summarization prompt ───────────────────────────────────
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Summarize the following conversation in 2-3 sentences. "
     "Keep all important facts like names, goals, locations. "
     "Be concise."),
    ("human", "{conversation}")
])
summary_chain=summary_prompt|llm|parser
summaries={}
history={}


In [4]:
def update_summary(session_id,human_msg,ai_msg):
    current_summary=summaries.get(session_id,"no conversation yet")
    conversation = (
    f"Current summary: {current_summary}\n\n"
    f"New exchange:\n"
    f"Human: {human_msg}\n"
    f"AI: {ai_msg}"
    )
    new_summary=summary_chain.invoke({"conversation":conversation})
    summaries[session_id]=new_summary
    return new_summary
    

In [9]:
def chat(message,session_id="session_1"):
    current_summary=summaries.get(session_id,"no conversation yet")
    response=chain.invoke({
    "input":message,
    "summary":current_summary
    })
    
    new_summary=update_summary(session_id,message,response)
    print(f"You    : {message}")
    print(f"AI     : {response}")
    print(f"Summary: {new_summary}")
    print()
    return response

In [10]:

chat("My name is Krish and I am from Ahmedabad Gujarat.")
chat("I have been studying ML for 8 months.")
chat("Now I am learning LangChain RAG and LangGraph.")
chat("My goal is ₹50k AI engineer internship in Bangalore.")
chat("I prefer startups over big companies.")
chat("What do you know about me?")  

You    : My name is Krish and I am from Ahmedabad Gujarat.
AI     : Nice to meet you, Krish! I'm happy to learn more about you. Ahmedabad is a beautiful city in Gujarat, known for its rich cultural heritage and history. What brings you here today?
Summary: There is no conversation yet. Here's a 2-3 sentence summary of the conversation so far:

Krish, a resident of Ahmedabad, Gujarat, has initiated a conversation with an AI. The AI has introduced itself and expressed interest in learning more about Krish.

You    : I have been studying ML for 8 months.
AI     : That's great to hear that you've been studying Machine Learning (ML) for 8 months. What aspects of ML have you been focusing on, and do you have any specific goals or areas of interest that you'd like to explore further?
Summary: Here's a 2-3 sentence summary of the conversation:

Krish, a resident of Ahmedabad, Gujarat, has been studying Machine Learning (ML) for 8 months. He has expressed interest in pursuing his ML knowledge f

'Based on our conversation so far, I know that you are from Bangalore and are interested in securing a ₹50k AI engineer internship in the city. You are seeking guidance on how to achieve this goal. Is there anything specific you would like to know or discuss further?'